## Text Cleaning

In [1]:
# import libraries
import pandas as pd
import numpy as np
import re
import nltk

In [2]:
# load csv
dataset = pd.read_csv('mail_data.csv')

print(dataset.head())
print(dataset.shape)

  Category                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...
(5572, 2)


In [3]:
# check missing values
print(dataset.isnull().sum())

Category    0
Message     0
dtype: int64


In [4]:
# lowercase 
dataset['Message'] = dataset['Message'].apply(lambda m: m.lower())

print(dataset.head())

  Category                                            Message
0      ham  go until jurong point, crazy.. available only ...
1      ham                      ok lar... joking wif u oni...
2     spam  free entry in 2 a wkly comp to win fa cup fina...
3      ham  u dun say so early hor... u c already then say...
4      ham  nah i don't think he goes to usf, he lives aro...


In [5]:
# remove numbers, punctuation, and special characters
dataset['Message'] = dataset['Message'].apply(lambda m: re.sub(r'[^A-Za-z]', ' ', m))

print(dataset.head())

  Category                                            Message
0      ham  go until jurong point  crazy   available only ...
1      ham                      ok lar    joking wif u oni   
2     spam  free entry in   a wkly comp to win fa cup fina...
3      ham  u dun say so early hor    u c already then say   
4      ham  nah i don t think he goes to usf  he lives aro...


In [6]:
# tokenization
from nltk.tokenize import word_tokenize

dataset['Message'] = dataset['Message'].apply(word_tokenize)

print(dataset.head())

  Category                                            Message
0      ham  [go, until, jurong, point, crazy, available, o...
1      ham                     [ok, lar, joking, wif, u, oni]
2     spam  [free, entry, in, a, wkly, comp, to, win, fa, ...
3      ham  [u, dun, say, so, early, hor, u, c, already, t...
4      ham  [nah, i, don, t, think, he, goes, to, usf, he,...


In [7]:
# remove stop words
from nltk.corpus import stopwords

stop_words = stopwords.words('english')
dataset['Message'] = dataset['Message'].apply(lambda tokens: [word for word in tokens if word not in stop_words])

print(dataset.head())

  Category                                            Message
0      ham  [go, jurong, point, crazy, available, bugis, n...
1      ham                     [ok, lar, joking, wif, u, oni]
2     spam  [free, entry, wkly, comp, win, fa, cup, final,...
3      ham      [u, dun, say, early, hor, u, c, already, say]
4      ham     [nah, think, goes, usf, lives, around, though]


In [8]:
# lemmatization
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

def get_wordnet_pos(pos_tag):
    first_letter = pos_tag[0]

    if first_letter == 'J':
        return wordnet.ADJ
    elif first_letter == 'V':
        return wordnet.VERB
    elif first_letter == 'R':
        return wordnet.ADV
    else:
        return wordnet.NOUN
    
lemmatizer = WordNetLemmatizer()

dataset['Message'] = dataset['Message'].apply(lambda tokens: [ 
    lemmatizer.lemmatize(word, get_wordnet_pos(pos_tag)) 
    for (word, pos_tag) in nltk.pos_tag(tokens)
    ])

print(dataset.head())

  Category                                            Message
0      ham  [go, jurong, point, crazy, available, bugis, n...
1      ham                     [ok, lar, joking, wif, u, oni]
2     spam  [free, entry, wkly, comp, win, fa, cup, final,...
3      ham      [u, dun, say, early, hor, u, c, already, say]
4      ham        [nah, think, go, usf, life, around, though]


In [9]:
# join words
dataset['Message'] = dataset['Message'].apply(lambda tokens: ' '.join(tokens))

print(dataset.head())

  Category                                            Message
0      ham  go jurong point crazy available bugis n great ...
1      ham                            ok lar joking wif u oni
2     spam  free entry wkly comp win fa cup final tkts st ...
3      ham                u dun say early hor u c already say
4      ham                nah think go usf life around though


In [10]:
# X and y
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

X = dataset['Message']
y = le.fit_transform(dataset['Category'])
print(X)

0       go jurong point crazy available bugis n great ...
1                                 ok lar joking wif u oni
2       free entry wkly comp win fa cup final tkts st ...
3                     u dun say early hor u c already say
4                     nah think go usf life around though
                              ...                        
5567    nd time try contact u u pound prize claim easy...
5568                               b go esplanade fr home
5569                                 pity mood suggestion
5570    guy bitch act like interested buying something...
5571                                       rofl true name
Name: Message, Length: 5572, dtype: object


In [11]:
# train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train)

1978    reply win weekly fifa world cup hold send stop...
3989    hello sort town already dont rush home eat nac...
3935                        come guoyang go n tell u tell
4078    hey sathya till dint meet even single time saw...
4086    orange bring ringtones time chart hero free hi...
                              ...                        
3772    hi wlcome back wonder get eaten lion something...
5191                                     sorry call later
5226                   prabha soryda realy frm heart sory
5390                               nt joke seriously tell
860                               say somebody name tampa
Name: Message, Length: 4457, dtype: object


## TF-IDF

In [12]:
# tf-idf
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)